In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import pypsa
import xlsxwriter
import tz_pypsa
import tz_pypsa.wrangle as wrangle
import pandas as pd
# import tz_solve
import plotly.express as px
import plotly.graph_objects as go
from tz_pypsa.model import Model
from tz_pypsa.utils import get_examples
import os
import glob

In [2]:
n_tp3_70 = pypsa.Network()
n_tp3_70.import_from_netcdf("C:/Users/jy/TransitionZero/Google - CFE - Documents/04. Country Specific Vault/Japan/02. Data & Results/Outputs/1_Diagnosis/TP3/Run006/JPN_P3_JPN03_006/solved_networks/hourly_matching_CFE70_2030.nc")

INFO:pypsa.io:Imported network hourly_matching_CFE70_2030.nc has buses, carriers, generators, links, loads, storage_units


In [3]:
n_tp2_70 = pypsa.Network()
n_tp2_70.import_from_netcdf("C:/Users/jy/TransitionZero/Google - CFE - Documents/04. Country Specific Vault/Japan/02. Data & Results/Outputs/1_Diagnosis/TP2/Run003/P2_JPN03_003/solved_networks/hourly_matching_CFE70_2030.nc")

INFO:pypsa.io:Imported network hourly_matching_CFE70_2030.nc has buses, carriers, generators, links, loads, storage_units


In [25]:
n_tp3_70_updated = pypsa.Network()
n_tp3_70_updated.import_from_netcdf("C:/Users/jy/TransitionZero/Google - CFE - Documents/04. Country Specific Vault/Japan/02. Data & Results/Outputs/1_Diagnosis/TP3/Run_007_JPN03/hourly_matching_CFE70_2030.nc")

INFO:pypsa.io:Imported network hourly_matching_CFE70_2030.nc has buses, carriers, generators, links, loads, storage_units


In [26]:
wrangle.get_scenario_emission_intensity(n_tp3_70_updated, "JPN03")

(166.57499139015138, 9785312.812)

In [ ]:
wrangle.get_scenario_emission_intensity(n_tp3_70, "JPN03")

(135.80205239537182, 9785312.812)

In [6]:
wrangle.get_scenario_emission_intensity(n_tp2_70, "JPN03")

(166.5748536656686, 9785312.812)

In [9]:
ci_greenfield_emissions_output = (n_tp3_70.generators_t.p[n_tp3_70.generators[n_tp3_70.generators.index.str.contains("C&I")].index]            
                            / n_tp3_70.generators[n_tp3_70.generators.index.str.contains("C&I")].efficiency
                            * n_tp3_70.generators[n_tp3_70.generators.index.str.contains("C&I")].carrier.map(n_tp3_70.carriers.co2_emissions)
                            ).fillna(0)

In [13]:
n_tp3_70.generators[n_tp3_70.generators.index.str.contains("C&I")].efficiency

Generator
JPN03 C&I Grid-solar-unspecified-ext-2030-PPA-Clean                     0.00
JPN03 C&I Grid-onshorewind-unspecified-ext-2030-PPA-Clean               0.00
JPN03 C&I Grid-blending-blueH2-gas-unspecified-ext-2030-PPA-Clean       0.55
JPN03 C&I Grid-blending-blueH2-gas-unspecified-ext-2030-PPA-Fossil      0.55
JPN03 C&I Grid-blending-blueNH3-coal-unspecified-ext-2030-PPA-Clean     0.38
JPN03 C&I Grid-blending-blueNH3-coal-unspecified-ext-2030-PPA-Fossil    0.38
JPN03 C&I Grid-gas-CCS-unspecified-ext-2030-PPA-Fossil                  0.46
JPN03 C&I Grid-gas-CCS-unspecified-ext-2030-PPA-Clean                   0.46
Name: efficiency, dtype: float64

In [18]:
n_tp3_70.generators_t.p[n_tp3_70.generators[n_tp3_70.generators.index.str.contains("C&I")].index]

Generator,JPN03 C&I Grid-solar-unspecified-ext-2030-PPA-Clean,JPN03 C&I Grid-onshorewind-unspecified-ext-2030-PPA-Clean,JPN03 C&I Grid-blending-blueH2-gas-unspecified-ext-2030-PPA-Clean,JPN03 C&I Grid-blending-blueH2-gas-unspecified-ext-2030-PPA-Fossil,JPN03 C&I Grid-blending-blueNH3-coal-unspecified-ext-2030-PPA-Clean,JPN03 C&I Grid-blending-blueNH3-coal-unspecified-ext-2030-PPA-Fossil,JPN03 C&I Grid-gas-CCS-unspecified-ext-2030-PPA-Fossil,JPN03 C&I Grid-gas-CCS-unspecified-ext-2030-PPA-Clean
snapshot,,,,,,,,
2030-01-01 00:00:00,0.0,749.067666,0.003058,0.027526,0.000931,0.003723,0.005773,0.013470
2030-01-01 01:00:00,0.0,773.076249,0.003063,0.027570,0.000924,0.003697,0.005745,0.013406
2030-01-01 02:00:00,0.0,770.675390,0.003060,0.027536,0.000911,0.003645,0.005736,0.013383
2030-01-01 03:00:00,0.0,809.089121,0.003059,0.027535,0.000909,0.003635,0.005731,0.013373
2030-01-01 04:00:00,0.0,801.886546,0.003058,0.027525,0.000908,0.003633,0.005732,0.013375
...,...,...,...,...,...,...,...,...
2030-12-31 19:00:00,0.0,969.946617,0.003067,0.027606,0.000907,0.003627,0.005703,0.013308
2030-12-31 20:00:00,0.0,950.739752,0.003071,0.027638,0.000909,0.003636,0.005700,0.013299
2030-12-31 21:00:00,0.0,960.343185,0.003073,0.027655,0.000910,0.003639,0.005697,0.013294


In [22]:
n_tp3_70.generators_t.p[n_tp3_70.generators[n_tp3_70.generators.index.str.contains("C&I")].index]/ n_tp3_70.generators[n_tp3_70.generators.index.str.contains("C&I")].efficiency* n_tp3_70.generators[n_tp3_70.generators.index.str.contains("C&I")].carrier.map(n_tp3_70.carriers.co2_emissions)

Generator,JPN03 C&I Grid-solar-unspecified-ext-2030-PPA-Clean,JPN03 C&I Grid-onshorewind-unspecified-ext-2030-PPA-Clean,JPN03 C&I Grid-blending-blueH2-gas-unspecified-ext-2030-PPA-Clean,JPN03 C&I Grid-blending-blueH2-gas-unspecified-ext-2030-PPA-Fossil,JPN03 C&I Grid-blending-blueNH3-coal-unspecified-ext-2030-PPA-Clean,JPN03 C&I Grid-blending-blueNH3-coal-unspecified-ext-2030-PPA-Fossil,JPN03 C&I Grid-gas-CCS-unspecified-ext-2030-PPA-Fossil,JPN03 C&I Grid-gas-CCS-unspecified-ext-2030-PPA-Clean
snapshot,,,,,,,,
2030-01-01 00:00:00,NaN,NaN,0.0,0.009909,0.0,0.003527,0.002485,0.0
2030-01-01 01:00:00,NaN,NaN,0.0,0.009925,0.0,0.003503,0.002473,0.0
2030-01-01 02:00:00,NaN,NaN,0.0,0.009913,0.0,0.003453,0.002469,0.0
2030-01-01 03:00:00,NaN,NaN,0.0,0.009912,0.0,0.003444,0.002467,0.0
2030-01-01 04:00:00,NaN,NaN,0.0,0.009909,0.0,0.003442,0.002467,0.0
...,...,...,...,...,...,...,...,...
2030-12-31 19:00:00,NaN,NaN,0.0,0.009938,0.0,0.003437,0.002455,0.0
2030-12-31 20:00:00,NaN,NaN,0.0,0.009950,0.0,0.003445,0.002453,0.0
2030-12-31 21:00:00,NaN,NaN,0.0,0.009956,0.0,0.003447,0.002452,0.0


In [10]:
ci_greenfield_emissions_output

Generator,JPN03 C&I Grid-solar-unspecified-ext-2030-PPA-Clean,JPN03 C&I Grid-onshorewind-unspecified-ext-2030-PPA-Clean,JPN03 C&I Grid-blending-blueH2-gas-unspecified-ext-2030-PPA-Clean,JPN03 C&I Grid-blending-blueH2-gas-unspecified-ext-2030-PPA-Fossil,JPN03 C&I Grid-blending-blueNH3-coal-unspecified-ext-2030-PPA-Clean,JPN03 C&I Grid-blending-blueNH3-coal-unspecified-ext-2030-PPA-Fossil,JPN03 C&I Grid-gas-CCS-unspecified-ext-2030-PPA-Fossil,JPN03 C&I Grid-gas-CCS-unspecified-ext-2030-PPA-Clean
snapshot,,,,,,,,
2030-01-01 00:00:00,0.0,0.0,0.0,0.009909,0.0,0.003527,0.002485,0.0
2030-01-01 01:00:00,0.0,0.0,0.0,0.009925,0.0,0.003503,0.002473,0.0
2030-01-01 02:00:00,0.0,0.0,0.0,0.009913,0.0,0.003453,0.002469,0.0
2030-01-01 03:00:00,0.0,0.0,0.0,0.009912,0.0,0.003444,0.002467,0.0
2030-01-01 04:00:00,0.0,0.0,0.0,0.009909,0.0,0.003442,0.002467,0.0
...,...,...,...,...,...,...,...,...
2030-12-31 19:00:00,0.0,0.0,0.0,0.009938,0.0,0.003437,0.002455,0.0
2030-12-31 20:00:00,0.0,0.0,0.0,0.009950,0.0,0.003445,0.002453,0.0
2030-12-31 21:00:00,0.0,0.0,0.0,0.009956,0.0,0.003447,0.002452,0.0
